# 知识

## 滑点
**滑点**（Slippage）是指预期或指令的价格与实际成交价格之间出现的差异。通常由以下因素导致：
- 市场波动性高： 当市场剧烈波动时，从下达指令到订单实际在交易所执行的期间内，价格可能已经发生了变动。
- 流动性不足： 市场上没有足够的买家或卖家来匹配订单，即订单无法立即以期望的价格完全成交，从而需要以下一个可用的价格来完成交易。
- 网络延迟： 交易指令从设备传输到交易服务器，再到交易所，这个过程中存在网络延迟。
- 订单类型：
  - **市价单**（Market Order）：以当前市场最佳可得价格立即执行。
    - 由于市场价格在不断变化，容易出现滑点。
  - **止损单**（Stop-Loss Order）：为了限制损失而设置，当股票价格达到止损价时会触发一个市价单。
    - 如果市场快速下跌，止损单可能无法在设定的价格上成交，而是以更低的价格成交，造成更大的损失。
  - **限价单**（Limit Order）：限价单指令只会在设定的价格或更好的价格成交。
    - 限价单通常不会出现不利的滑点，但缺点是如果价格未能达到您的设定，订单可能无法成交。

# 订单处理

## check_group_lens_nb
生成未成交订单结果对象

参数
- `status`：int，订单状态码，通常表示拒绝或忽略
  - OrderStatus.`Rejected`: 订单被拒绝  
  - OrderStatus.`Ignored`: 订单被忽略
- `status_info`：int，具体的状态信息码，说明拒绝原因
  - OrderStatusInfo.`NoCashLong`：做多资金不足
  - OrderStatusInfo.`NoOpenPosition`：无持仓可平
  - OrderStatusInfo.`SizeZero`：订单大小为零
  - OrderStatusInfo.`MaxSizeExceeded`：超过最大订单限制
  - OrderStatusInfo.`MinSizeNotReached`：未达到最小订单限制
  - OrderStatusInfo.`CantCoverFees`：无法承担手续费
  - OrderStatusInfo.`PartialFill`：部分成交被拒绝

返回：订单结果对象 OrderResult。包含以下字段
- size: np.nan (未成交数量)
- price: np.nan (未成交价格) 
- fees: np.nan (未产生手续费)
- side: -1 (无交易方向)
- status: 传入的状态码
- status_info: 传入的状态详情码

### 源码
```python
@njit(cache=True)
def order_not_filled_nb(status: int, status_info: int) -> OrderResult:
    return OrderResult(np.nan, np.nan, np.nan, -1, status, status_info)
```

## buy_nb
执行买入订单或平空头操作。

### 参数

- `exec_state` : ExecuteOrderState，当前执行状态。包含：
  - `cash`: 总现金。通常是指多个资产，包括空头的保证金
  - `position`: 当前头寸（正数=多头，负数=空头）。针对当前资产
  - `debt`: 空头债务（用于计算平均成本）。针对当前资产，如果其空头，指的是 *空头数 * 空头时成交价*
  - `free_cash`: 可用现金。可供当前资产交易的自由现金
- `size` : float，期望买入数量。可以是：
  - 正数: 具体买入数量
  - np.inf: 使用所有可用资金买入
- `price` : float，目标买入价格。
  - 实际成交价会考虑滑点调整
- `direction` : int，交易方向限制：
    - `Direction.Both`: 允许开多头或平空头
    - `Direction.LongOnly`: 只允许开多头
    - `Direction.ShortOnly`: 只允许平空头
- `fees` : float, 比例手续费率（例如 0.001 表示 0.1%）
- `fixed_fees` : float, 固定手续费（绝对金额）
- `slippage` : float, 滑点率。
  - 买入时向上滑点，实际价格 = price * (1 + slippage)
- `min_size` : float, 最小订单数量。小于此值的订单将被拒绝
- `max_size` : float, 最大订单数量。超过此值的订单将被截断或拒绝
- `size_granularity` : float, 数量粒度。
  - 订单数量将向下取整到此粒度的整数倍。例如：粒度为 0.1，则 1.37 会变为 1.3
- `lock_cash` : bool, 是否锁定现金。
  - 如果为 True：
    - 多头时只能使用 free_cash
    - 空头时需考虑平仓所需资金
- `allow_partial` : bool, 是否允许部分成交。
  - 为 False 时，资金不足的订单将被完全拒绝
- `percent` : float, 资金使用比例。限制最多使用多少比例的可用资金
    
返回：tuple[ExecuteOrderState, OrderResult]，新的执行状态和订单结果
- `ExecuteOrderState`: 更新后的投资组合状态
  - `cash`: 扣除交易成本后的现金
  - `position`: 更新后的头寸
  - `debt`: 更新后的空头债务
  - `free_cash`: 更新后的可用现金
- `OrderResult`: 订单执行结果
  - `size`: 实际成交数量
  - `price`: 实际成交价格（含滑点）
  - `fees`: 实际支付的手续费
  - `side`: OrderSide.Buy
  - `status`: OrderStatus.Filled 或相应的拒绝状态
  - `status_info`: 详细状态信息

### 逻辑

注意：
- 空头时，会锁定 *2倍的空头数×当时成交价* 作为保证金，防止无资金平空头

计算**调整价格** `adj_price = price * (1 + slippage)`
- 因为延时，下达订单时的价格与最终成交时的价格存在差异

计算资金限制 `cash_limit`
- 参数 `lock_cash`
  - 为 `False`：可用所有现金 `cash_limit = exec_state.cash`
  - 为 `True`（只考虑使用 `free_cash` 以及空头时被锁定的保证金）
    - 当前多头 `exec_state.position >= 0`
      - 此时 `cash_limit = exec_state.free_cash`
    - 当前空头
      - 计算完全平仓需要多少现金 `cover_req_cash`
        - $空头总数 \cdot 调整价格 \cdot \left( {1 + 比例手续费率} \right) + 固定手续费$
      - 计算完全平仓后的自由现金 `cover_free_cash=exec_state.free_cash + 2 * exec_state.debt, -cover_req_cash`
        - $当前自由现金  + 释放的保证金 - 完全平仓成本$
      - 如果 `cover_free_cash > 0`（有足够现金平掉当前全部空头）
        - 可用所有现金 `cash_limit = exec_state.free_cash + 2 * exec_state.debt`
      - 如果 `cover_free_cash < 0`
        - 计算空头的平均入场价格 `avg_entry_price = exec_state.debt / abs(exec_state.position)`
        - 计算最多能平空头数 `max_short_size`
          - $自由现金 + 2 \cdot 平均入场价格 \cdot x = 调整价格\left( {1 + 比例手续费率} \right)x + 固定手续费$
        - 资金限制 `cash_limit = max_short_size * adj_price * (1 + fees) + fixed_fees`
      - 否则（即 `cover_free_cash == 0`）
        - `cash_limit=exec_state.free_cash + 2 * exec_state.debt`

考虑参数 `percent` 即比例限制
- `cash_limit = min(cash_limit, percent * cash_limit)`

如果是下述情况，生成未成交订单对象
- 允许开多头
  - `cash_limit = 0`，即无可用现金
  - 期望买入数量 `size` 和 `cash_limit` 都是 `inf`
- 只允许平空头
  - `exec_state.position == 0`：当前无头寸，无法进行平仓操作

计算调整后的订单大小 `adj_size`
- 只允许平空头 
  - `adj_size = min(-exec_state.position, size)`，即订单大小不能超过当前空头头寸的绝对值
- 允许开多头
  - `adj_size = size`
- 根据粒度调整 `adj_size = adj_size // size_granularity * size_granularity`
  - 例如：粒度为 0.1，1.37——>13——>1.3

计算完成此订单的所需现金总额 `total_req_cash`
- `total_req_cash = adj_size * adj_price * (1 + fees) + req_fees`

检查资金是否充足
- 如果 `total_req_cash <= cash_limit`
  - 最终成交数量 `final_size = adj_size `
  - 最终实际支付手续费 `fees_paid = adj_size * adj_price * fees + fixed_fees`
  - 最终实际使用现金 `final_req_cash = total_req_cash`
- 否则（需要减少订单数量以适用资金数 `cash_limit`）
  - 计算 `max_req_cash = (cash_limit - fixed_fees) / (1 + fees)`
    - 如果 `max_req_cash < 0` 即固定手续费都无法承担，返回未成交订单对象
  - 计算最大可购买数 `max_acq_size = max_req_cash / adj_price`
  - 根据粒度 `size_granularity` 调整最大可购买数
  - 确定最终成交数量 `final_size`、实际支付手续费 `fees_paid`、实际使用现金 `final_req_cash`
  
检查，如果是下述情况，返回未成交订单对象
- `adj_size < 0`
- `final_size` 小于参数 `min_size`
- 参数 `size < ∞` 并且 `final_size < size` 并且参数 `allow_partial==False` 

更新
- 现金：`new_cash = exec_state.cash - final_req_cash`
- 头寸：`new_position = exec_state.position + final_size`
- 如果原来是空头 `exec_state.position < 0`
  - 计算买入数量 `short_size`
  - 更新债务 `new_debt`$ =exec\_state.debt - short\_size\frac{{exec\_state.debt}}{{\left| {exec\_state.position} \right|}}$
  - 更新新自由现金 `new_free_cash`：$原自由现金 + 释放的债务保证金(2倍) - 交易成本$
    - $exec\_state.free\_cash + 2short\_size\frac{{exec\_state.debt}}{{\left| {exec\_state.position} \right|}} - final\_req\_cash$
- 如果原来无空头或者为多头
  - 更新债务 `new_debt = exec_state.debt`
  - 更新新自由现金 `new_free_cash = exec_state.free_cash - final_req_cash`

构建
- 订单结果 `OrderResult`
  - `final_size`：实际成交数量
  - `adj_price`：实际成交价格（含滑点）
  - `fees_paid`：实际支付的手续费
  - `OrderSide.Buy`：订单方向：买入
  - `OrderStatus.Filled`：订单状态：已成交
  - `-1`：状态详情，无特殊信息
- 执行订单状态 `ExecuteOrderState`
  - `cash=new_cash`：更新后的现金余额
  - `position=new_position`：更新后的头寸
  - `debt=new_debt`：更新后的债务
  - `free_cash=new_free_cash`：更新后的可用现金

## sell_nb
执行卖出订单或开空头操作。

### 参数
- `exec_state` : ExecuteOrderState，当前执行状态。包含：
  - `cash`: 总现金。通常是指多个资产，包括空头的保证金
  - `position`: 当前头寸（正数=多头，负数=空头）。针对当前资产
  - `debt`: 空头债务（用于计算平均成本）。针对当前资产，如果其空头，指的是 *空头数 * 空头时成交价*
  - `free_cash`: 可用现金。
- `size`：float，期望卖出数量。可以是：
  - 正数: 具体卖出数量
  - np.inf: 卖出所有持仓或开最大空头
- `price`：float，目标卖出价格。实际成交价会考虑滑点调整
- `direction`：int, 可选 (默认: Direction.Both)
  - `Direction.Both`：允许平多头或开空头
  - `Direction.LongOnly`：只允许平多头
  - `Direction.ShortOnly`：只允许开空头
- `fees`：float, 可选 (默认: 0.0)。比例手续费率（例如 0.001 表示 0.1%）
- `fixed_fees`：float, 可选 (默认: 0.0)。固定手续费（绝对金额）
- `slippage`：float, 可选 (默认: 0.0)，滑点率。
  - 卖出时向下滑点，实际价格 = price * (1 - slippage)
- `min_size`：float, 可选 (默认: 0.0)，最小订单数量。小于此值的订单将被拒绝
- `max_size`：float, 可选 (默认: np.inf)，最大订单数量。超过此值的订单将被截断或拒绝
- `size_granularity`：float, 可选 (默认: np.nan)，数量粒度。
  - 订单数量将向下取整到此粒度的整数倍
- `lock_cash`：bool, 可选 (默认: False)，是否锁定现金。
  - 如果为 True，则限制空头开仓的最大数量
- `allow_partial`：bool, 可选 (默认: True)，是否允许部分成交。
  - 为 False 时，保证金不足的订单将被完全拒绝
- `percent`：float, 可选 (默认: np.nan)，头寸使用比例。限制最多卖出多少比例的可卖数量

返回：tuple[ExecuteOrderState, OrderResult]，新的执行状态和订单结果
- `ExecuteOrderState`：更新后的投资组合状态
  - `cash`：增加卖出收入后的现金（平多头时）或不变（开空头时）
  - `position`：更新后的头寸（减少或变负）
  - `debt`：更新后的空头债务（开空头时增加）
  - `free_cash`：更新后的可用现金（考虑保证金锁定）
- `OrderResult`：订单执行结果
  - `size`：实际成交数量
  - `price`：实际成交价格（含滑点）
  - `fees`：实际支付的手续费
  - `side`：OrderSide.Sell
  - `status`：OrderStatus.Filled 或相应的拒绝状态
  - `status_info`：详细状态信息

### 逻辑

注意：
- 空头时，会锁定 *2倍的空头数×当时成交价* 作为保证金，防止无资金平空头

计算**调整价格** `adj_price = price * (1 - slippage)`
- 因为延时，下达订单时的价格与最终成交时的价格存在差异

计算可卖出的最大数量 `size_limit`
- 只允许多头（参数 `direction == Direction.LongOnly`）
  - 卖出数量不能超过当前多头头寸 `size_limit = min(exec_state.position, size)`
- 允许开空头
  - 如果 `lock_cash=True` 或者 `size` 是 `inf` 以及 `percent` 是 `Nan`
    - 计算总可用现金 `total_free_cash`
      - $自由现金 + \max \left( {头寸数,0} \right) \cdot 调整价格\left( {1 - 比例手续费率} \right)$
    - 如果 `total_free_cash <= 0`
      - 如果当前头寸数 `exec_state.position <= 0`：返回未成交订单对象
      - 只能平现有多头：`max_size_limit = max(exec_state.position, 0)  `
    - 否则
      - 计算最大可开空头数量 `max_short_size` $= \frac{{总可用现金 - 固定手续费}}{{调整价格\left( {1 + 比例手续费率} \right)}}$
        - 开空头只需要保证金与手续费，这里这样限制是为了防止过度杠杆即之后没现金平空头
      - `max_size_limit =  max(exec_state.position, 0) + max_short_size`
      - 如果 `max_size_limit <= 0`：返回未成交订单对象
    - 如果 `lock_cash=True`
      - 如果 `size` 是 `inf` 并且 `percent` 不是 `Nan`
        - `size_limit = min(percent * max_size_limit, max_size_limit)`
        - `percent = np.nan`
      - 否则如果 `percent` 不是 `Nan`
        - `size_limit = min(percent * size, max_size_limit)`
        - `percent = np.nan`
      - 否则（`percent` 是 `Nan`）
        - `size_limit = max_size_limit`

  - 否则
    - 使用原始订单大小 `size_limit = size`

考虑参数 `percent` 即比例限制
- 如果 `precent` 不是 `Nan`：`size_limit = percent * size_limit`

考虑最大订单限制
- 如果 `size_limit > max_size` 
  - 如果允许部分成交 `allow_partial`：`size_limit = max_size`
  - 否则，生成未成交订单对象

如果允许开空头且 `size_limit` 为 `inf`：即无限大空头开仓，报错

如果只允许平多头且当前无头寸：生成未成交订单对象

根据粒度调整 `size_limit = size_limit // size_granularity * size_granularity`
- 例如：粒度为 0.1，1.37——>13——>1.3

检查，如果是下述情况，生成未成交订单对象
- `size_limit < 0`
- `size_limit < min_size`
- `size` 有限且 `size_limit < size` 且不允许部分成交

计算卖出获得的总现金 `acq_cash`、手续费 `fees_paid`、扣除手续费后的现金收入
- `acq_cash = size_limit * adj_price`
- `fees_paid = acq_cash * fees + fixed_fees`
- `final_acq_cash = acq_cash - fees_paid`

更新
- 现金：`new_cash = exec_state.cash + final_acq_cash`
- 头寸：`new_position = exec_state.position - size_limit`
- 如果变成空头 `new_position < 0`
  - 计算空头数量 `short_size`
    - 原来是空头：`short_size = size_limit`（原来就是空头，增加空头规模）
    - 否则：`short_size = abs(new_position)`
  - 更新债务 `new_debt`$ =exec\_state.debt + short\_value$
  - 更新新自由现金 `new_free_cash`：$原自由现金 - 2倍空头价值 + final\_acq\_cash$
- 否则
  - 更新债务 `new_debt = exec_state.debt`
  - 更新新自由现金 `new_free_cash = exec_state.free_cash + final_acq_cash`

构建
- 订单结果 `OrderResult`
  - `final_size`：实际成交数量
  - `adj_price`：实际成交价格（含滑点）
  - `fees_paid`：实际支付的手续费
  - `OrderSide.Buy`：订单方向：卖出
  - `OrderStatus.Filled`：订单状态：已成交
  - `-1`：状态详情，无特殊信息
- 执行订单状态 `ExecuteOrderState`
  - `cash=new_cash`：更新后的现金余额
  - `position=new_position`：更新后的头寸
  - `debt=new_debt`：更新后的债务
  - `free_cash=new_free_cash`：更新后的可用现金

## execute_order_nb
根据订单 `order: Order` 和投资组合状态 `state: ProcessOrderState`执行 `buy_nb` 或 `sell_nb`。

### 参数和返回
参数
- `state`：ProcessOrderState。当前的投资组合处理状态
  - `cash`：float，现金余额：当前列或现金共享组的现金总额
  - `position`：float，持仓数量：当前列的资产持仓数量（正数多头，负数空头）
  - `debt`：float，做空债务：当前列做空操作产生的债务总额
  - `free_cash`：float，可用现金：当前列或现金共享组的可用于交易的现金
  - `val_price`：float，估值价格：当前列资产的估值价格（用于价值计算）
  - `value`：float，总价值：当前列或现金共享组的总价值（现金+持仓价值-债务）
  - `oidx`：int，单索引：对应的订单记录在order_records数组中的索引位置
  - `lidx`：int，日志索引：对应的日志记录在log_records数组中的索引位置   
- `order`：Order，待执行的订单对象
  - `size`：float = np.inf，订单大小：要交易的数量或金额
  - `price`：float = np.inf，订单价格：每单位的交易价格
  - `size_type`：int = SizeType.Amount，大小类型：订单大小的解释方式
  - `direction`：int = Direction.Both，允许方向：订单允许的交易方向
  - `fees`：float = 0.0，手续费率：按订单价值的百分比收费
  - `fixed_fee`：float = 0.0，固定手续费：每笔订单的固定费用
  - `slippage`：float = 0.0，滑点率：价格滑动的百分比
  - `min_size`：float = 0.0，最小大小：订单的最小允许大小
  - `max_size`：float = np.inf，最大大小：订单的最大允许大小
  - `size_granularity`：float = np.nan，大小粒度：订单大小的最小调整单位
  - `reject_prob`：float = 0.0，拒绝概率：随机拒绝订单的概率
  - `lock_cash`：bool = False，锁定现金：做空时是否锁定现金
  - `allow_partial`：bool = True，允许部分成交：是否接受部分填充
  - `raise_reject`：bool = False，拒绝异常：拒绝时是否抛出异常
  - `log`：bool = False，日志记录：是否记录此订单的详细日志
  
返回：tp.Tuple[ExecuteOrderState, OrderResult]，订单执行状态和订单结果的元组
- 执行订单状态 `ExecuteOrderState`
  - `cash`：float，现金余额：订单执行后的现金总额
  - `position`：float，持仓数量：订单执行后的持仓数量
  - `debt`：float，做空债务：订单执行后的债务总额
  - `free_cash`：float，可用现金：订单执行后的可用现金
- 订单结果 `OrderResult`
  - `size`：float，实际成交大小
  - `price`：float，实际成交价格（含滑点调整）
  - `fees`：float，实际支付的总手续费
  - `side`：int，实际执行的订单方向（OrderSide枚举）
  - `status`：int，订单执行状态（OrderStatus枚举）
  - `status_info`：int，订单状态详细信息（OrderStatusInfo枚举）

### 逻辑
（1）获取投资组合状态 `state` 中的各成分，接近零时设为精确零
- `cash`、`position`、`debt`、`free_cash`、`val_price`、`value`

（2）预构建订单执行状态 `exec_state`
- `cash`、`position`、`debt`、`free_cash`

（3）检查订单 `order` 和投资组合状态 `state` 中各成分的合法性

（4）计算订单大小 `order_size`
- `order_size = order.size`，订单大小类型 `order_size_type = order.size_type`
- 如果是只做空订单（`order.direction == Direction.ShortOnly`）
  - `order_size *= -1`
  - 这是为了表达如下的语义：正数表示增加头寸，负数表示减少头寸
- 如果 `order_size_type` 为 `SizeType.TargetPercent`（目标百分比模式）
  - 转换为目标价值
  - `order_size *= value`
  - `order_size_type = SizeType.TargetValue`
- 如果 `order_size_type` 为 `SizeType.Value` 或者 `SizeType.TargetValue`
  - 将价值转换为数量：`order_size /= val_price`
  - 如果 `order_size_type` 为 `SizeType.Value`，则变为 `SizeType.Amount`，否则 `SizeType.TargetAmount`
- 如果 `order_size_type` 为 `SizeType.TargetAmount`
  - 计算需要交易的数量 = 目标数量 - 当前持有数量
  - `order_size -= position`
  - `order_size_type = SizeType.Amount` 
- 如果 `order_size_type` 为 `SizeType.Percent`（基于当前可用资源的百分比）
  - `percent = abs(order_size)`，获取去符号的百分比值
  - `order_size = np.sign(order_size) * np.inf`，设置为带符号的无限大
  - `order_size_type = SizeType.Amount`

（5）根据 `order_size` 的符号执行买入或卖出操作，并返回相应结果
- `order_size > 0`：执行 `buy_nb`
- `order_size <= 0`：执行 `sell_nb`
- 参数
  - `exec_state`：当前执行状态
  - `+/-order_size`：买入/卖出数量
  - `order.price`：买入/卖出价格
  - `direction=order.direction`：交易方向限制
  - `fees=order.fees`：手续费率
  - `fixed_fees=order.fixed_fees`：固定手续费
  - `slippage=order.slippage`：滑点设置
  - `min_size=order.min_size`：最小订单大小
  - `max_size=order.max_size`：最大订单大小
  - `size_granularity=order.size_granularity`：数量粒度
  - `lock_cash=order.lock_cash`：是否锁定现金
  - `allow_partial=order.allow_partial`：是否允许部分成交
  - `percent=percent`：资金使用百分比

## fill_log_record_nb

将订单执行过程中的所有关键信息记录到日志记录中 `record`。

参数：
- `record`：Record，待填充的日志记录对象
- `record_id`：int，记录唯一标识符
- `i`：int，时间索引（通常是时间步或K线索引）
- `col`：int，列索引（资产标识符）
- `group`：int，组标识符（用于资产分组）
- `cash`：float，执行前的现金余额
- `position`：float，执行前的头寸
- `debt`：float，执行前的债务
- `free_cash`：float，执行前的可用现金
- `val_price`：float，执行前的估值价格
- `value`：float，执行前的组合价值
- `order`：Order，原始订单对象
- `new_cash`：float，执行后的现金余额
- `new_position`：float，执行后的头寸
- `new_debt`：float，执行后的债务
- `new_free_cash`：float，执行后的可用现金
- `new_val_price`：float，执行后的估值价格
- `new_value`：float，执行后的组合价值
- `order_result`：OrderResult，订单执行结果
  - `size`：float，实际成交大小
  - `price`：float，实际成交价格（含滑点调整）
  - `fees`：float，实际支付的总手续费
  - `side`：int，实际执行的订单方向（OrderSide枚举）
  - `status`：int，订单执行状态（OrderStatus枚举）
  - `status_info`：int，订单状态详细信息（OrderStatusInfo枚举）
- `order_id`：int，订单标识符

### 源码
```python
@njit(cache=True)
def fill_log_record_nb(record: tp.Record,
                       record_id: int,
                       i: int,
                       col: int,
                       group: int,
                       cash: float,
                       position: float,
                       debt: float,
                       free_cash: float,
                       val_price: float,
                       value: float,
                       order: Order,
                       new_cash: float,
                       new_position: float,
                       new_debt: float,
                       new_free_cash: float,
                       new_val_price: float,
                       new_value: float,
                       order_result: OrderResult,
                       order_id: int) -> None:

    record['id'] = record_id
    record['group'] = group
    record['col'] = col
    record['idx'] = i
    record['cash'] = cash
    record['position'] = position
    record['debt'] = debt
    record['free_cash'] = free_cash
    record['val_price'] = val_price
    record['value'] = value
    record['req_size'] = order.size
    record['req_price'] = order.price
    record['req_size_type'] = order.size_type
    record['req_direction'] = order.direction
    record['req_fees'] = order.fees
    record['req_fixed_fees'] = order.fixed_fees
    record['req_slippage'] = order.slippage
    record['req_min_size'] = order.min_size
    record['req_max_size'] = order.max_size
    record['req_size_granularity'] = order.size_granularity
    record['req_reject_prob'] = order.reject_prob
    record['req_lock_cash'] = order.lock_cash
    record['req_allow_partial'] = order.allow_partial
    record['req_raise_reject'] = order.raise_reject
    record['req_log'] = order.log
    record['new_cash'] = new_cash
    record['new_position'] = new_position
    record['new_debt'] = new_debt
    record['new_free_cash'] = new_free_cash
    record['new_val_price'] = new_val_price
    record['new_value'] = new_value
    record['res_size'] = order_result.size
    record['res_price'] = order_result.price
    record['res_fees'] = order_result.fees
    record['res_side'] = order_result.side
    record['res_status'] = order_result.status
    record['res_status_info'] = order_result.status_info
    record['order_id'] = order_id
```

## fill_order_record_nb
将成功执行的订单结果填充到订单记录 `record` 中。
- 与 `fill_log_record_nb` 不同，该函数只记录核心的订单执行结果，不包括详细的状态变化和请求参数。

参数
- `record`：Record，待填充的订单记录对象
- `record_id`：int，记录唯一标识符
- `i`：int，
- `col`：int，列索引（资产标识符）
- `order_result`：OrderResult，订单执行结果对象
  - `size`：float，实际成交大小
  - `price`：float，实际成交价格（含滑点调整）
  - `fees`：float，实际支付的总手续费
  - `side`：int，实际执行的订单方向（OrderSide枚举）
  - `status`：int，订单执行状态（OrderStatus枚举）
  - `status_info`：int，订单状态详细信息（OrderStatusInfo枚举）

### 源码
```python
@njit(cache=True)
def fill_order_record_nb(record: tp.Record,
                         record_id: int,
                         i: int,
                         col: int,
                         order_result: OrderResult) -> None:

    record['id'] = record_id
    record['col'] = col
    record['idx'] = i
    record['size'] = order_result.size
    record['price'] = order_result.price
    record['fees'] = order_result.fees
    record['side'] = order_result.side
```

## raise_rejected_order_nb
根据订单结果 `order_result: OrderResult` 抛出订单拒绝异常 `RejectedOrderError`。

### 源码
```python
@njit(cache=True)
def raise_rejected_order_nb(order_result: OrderResult) -> None:

    if order_result.status_info == OrderStatusInfo.SizeNaN:
        raise RejectedOrderError("Size is NaN")
    if order_result.status_info == OrderStatusInfo.PriceNaN:
        raise RejectedOrderError("Price is NaN")
    if order_result.status_info == OrderStatusInfo.ValPriceNaN:
        raise RejectedOrderError("Asset valuation price is NaN")
    if order_result.status_info == OrderStatusInfo.ValueNaN:
        raise RejectedOrderError("Asset/group value is NaN")
    if order_result.status_info == OrderStatusInfo.ValueZeroNeg:
        raise RejectedOrderError("Asset/group value is zero or negative")
    if order_result.status_info == OrderStatusInfo.SizeZero:
        raise RejectedOrderError("Size is zero")
    if order_result.status_info == OrderStatusInfo.NoCashShort:
        raise RejectedOrderError("Not enough cash to short")
    if order_result.status_info == OrderStatusInfo.NoCashLong:
        raise RejectedOrderError("Not enough cash to long")
    if order_result.status_info == OrderStatusInfo.NoOpenPosition:
        raise RejectedOrderError("No open position to reduce/close")
    if order_result.status_info == OrderStatusInfo.MaxSizeExceeded:
        raise RejectedOrderError("Size is greater than maximum allowed")
    if order_result.status_info == OrderStatusInfo.RandomEvent:
        raise RejectedOrderError("Random event happened")
    if order_result.status_info == OrderStatusInfo.CantCoverFees:
        raise RejectedOrderError("Not enough cash to cover fees")
    if order_result.status_info == OrderStatusInfo.MinSizeNotReached:
        raise RejectedOrderError("Final size is less than minimum allowed")
    if order_result.status_info == OrderStatusInfo.PartialFill:
        raise RejectedOrderError("Final size is less than requested")
    raise RejectedOrderError
```

## update_value_nb
更新估值价格和投资组合总价值
- 每次订单执行后，需要更新资产的估值价格和投资组合的总价值

参数
- `cash_before`：float，订单执行前的现金余额
- `cash_now`：float，订单执行后的现金余额  
- `position_before`：float，订单执行前的头寸数量
- `position_now`：float，订单执行后的头寸数量
- `val_price_before`：float，订单执行前的资产估值价格
- `price`：float，订单执行时的实际成交价格（用作新的估值价格）
- `value_before`：float，订单执行前的投资组合总价值

逻辑
- 新估值价格 = 订单最新成交价格 `price`
- 现金流变化 = 执行后现金 - 执行前现金
- 资产价值变化 = 新头寸 × 新价格 - 旧头寸 × 旧价格
- 新总价值 = 旧总价值 + 资产价值变化 + 现金流变化
    
返回：`tuple[float, float]` (新估值价格, 新总价值)

### 源码
```python
@njit(cache=True)
def update_value_nb(cash_before: float,
                    cash_now: float,
                    position_before: float,
                    position_now: float,
                    val_price_before: float,
                    price: float,
                    value_before: float) -> tp.Tuple[float, float]:
    # 更新估值价格为最新成交价格
    val_price_now = price
    
    # 计算现金流变化（现金的增减）
    cash_flow = cash_now - cash_before
    
    # 计算订单执行前的资产价值
    if position_before != 0:
        asset_value_before = position_before * val_price_before  # 旧头寸 * 旧价格
    else:
        asset_value_before = 0.  # 无头寸时资产价值为零
    
    # 计算订单执行后的资产价值
    if position_now != 0:
        asset_value_now = position_now * val_price_now  # 新头寸 * 新价格
    else:
        asset_value_now = 0.  # 无头寸时资产价值为零
    
    # 计算资产价值的变化
    asset_value_diff = asset_value_now - asset_value_before
    
    # 计算新的投资组合总价值
    # 新总价值 = 原总价值 + 现金变化 + 资产价值变化
    value_now = value_before + cash_flow + asset_value_diff
    
    return val_price_now, value_now
```

## process_order_nb
订单处理的最高层包装函数，整合了订单执行、记录保存和状态更新的完整流程

### 参数和返回
- `i`：int，时间索引（当前时间步或K线索引）
- `col`：int，资产列索引（资产标识符）
- `group`：int，资产组索引（用于分组管理）
- `state`：ProcessOrderState，当前的处理状态（包含现金、头寸、债务等信息）
  - `cash`：float，现金余额：当前列或现金共享组的现金总额
  - `position`：float，持仓数量：当前列的资产持仓数量（正数多头，负数空头）
  - `debt`：float，做空债务：当前列做空操作产生的债务总额
  - `free_cash`：float，可用现金：当前列或现金共享组的可用于交易的现金
  - `val_price`：float，估值价格：当前列资产的估值价格（用于价值计算）
  - `value`：float，总价值：当前列或现金共享组的总价值（现金+持仓价值-债务）
  - `oidx`：int，订单索引：对应的订单记录在 `order_records` 数组中的索引位置
  - `lidx`：int，日志索引：对应的日志记录在 `log_records` 数组中的索引位置
- `update_value`：bool，是否在订单成交后更新投资组合价值
- `order`：Order，待处理的订单对象
  - `size`：float = np.inf，订单大小：要交易的数量或金额
  - `price`：float = np.inf，订单价格：每单位的交易价格
  - `size_type`：int = SizeType.Amount，大小类型：订单大小的解释方式
  - `direction`：int = Direction.Both，允许方向：订单允许的交易方向
  - `fees`：float = 0.0，手续费率：按订单价值的百分比收费
  - `fixed_fees`：float = 0.0，固定手续费：每笔订单的固定费用
  - `slippage`：float = 0.0，滑点率：价格滑动的百分比
  - `min_size`：float = 0.0，最小大小：订单的最小允许大小
  - `max_size`：float = np.inf，最大大小：订单的最大允许大小
  - `size_granularity`：float = np.nan，大小粒度：订单大小的最小调整单位
  - `reject_prob`：float = 0.0，拒绝概率：随机拒绝订单的概率
  - `lock_cash`：bool = False，锁定现金：做空时是否锁定现金
  - `allow_partial`：bool = True，允许部分成交：是否接受部分填充
  - `raise_reject`：bool = False，拒绝异常：拒绝时是否抛出异常
  - `log`：bool = False，日志记录：是否记录此订单的详细日志
- `order_records`：RecordArray，用于存储成功订单记录的数组
- `log_records`：RecordArray，用于存储详细日志记录的数组
    
返回：`tuple[OrderResult, ProcessOrderState]` (订单执行结果, 更新后的处理状态)

### 流程
- 使用参数 `state: ProcessOrderState` 和 `order: Order` 调用 `execute_order_nb` 执行订单
  - 返回 `tp.Tuple[ExecuteOrderState, OrderResult]`
- 使用 `raise_rejected_order_nb` 检查订单执行结果，决定是否需要抛出订单拒绝异常 `RejectedOrderError`
- 如果订单成交且需要更新价值，使用 `update_value_nb` 更新估值价格和投资组合总价值
- 如果订单成交，则使用 `fill_order_record_nb` 将结果保存到 `order_records`
- 如果需要记录日志，则使用 `fill_log_record_nb` 将详细信息保存到 `log_records`
- 构造并返回更新后的处理状态 `tuple[OrderResult, ProcessOrderState]`

### 源码
```python
@njit(cache=True)
def process_order_nb(i: int,
                     col: int,
                     group: int,
                     state: ProcessOrderState,
                     update_value: bool,
                     order: Order,
                     order_records: tp.RecordArray,
                     log_records: tp.RecordArray) -> tp.Tuple[OrderResult, ProcessOrderState]:

    exec_state, order_result = execute_order_nb(state, order)

    is_rejected = order_result.status == OrderStatus.Rejected
    if is_rejected and order.raise_reject:
        raise_rejected_order_nb(order_result)

    is_filled = order_result.status == OrderStatus.Filled
    if is_filled and update_value:
        new_val_price, new_value = update_value_nb(
            state.cash,
            exec_state.cash,
            state.position,
            exec_state.position,
            state.val_price,
            order_result.price,
            state.value
        )
    else:
        new_val_price = state.val_price
        new_value = state.value

    new_oidx = state.oidx
    if is_filled:
        # Fill order record
        if state.oidx > len(order_records) - 1:
            raise IndexError("order_records index out of range. Set a higher max_orders.")
        fill_order_record_nb(
            order_records[state.oidx],
            state.oidx,
            i,
            col,
            order_result
        )
        new_oidx += 1

    new_lidx = state.lidx
    if order.log:
        # Fill log record
        if state.lidx > len(log_records) - 1:
            raise IndexError("log_records index out of range. Set a higher max_logs.")
        fill_log_record_nb(
            log_records[state.lidx],
            state.lidx,
            i,
            col,
            group,
            state.cash,
            state.position,
            state.debt,
            state.free_cash,
            state.val_price,
            state.value,
            order,
            exec_state.cash,
            exec_state.position,
            exec_state.debt,
            exec_state.free_cash,
            new_val_price,
            new_value,
            order_result,
            state.oidx if is_filled else -1
        )
        new_lidx += 1

    new_state = ProcessOrderState(
        cash=exec_state.cash,
        position=exec_state.position,
        debt=exec_state.debt,
        free_cash=exec_state.free_cash,
        val_price=new_val_price,
        value=new_value,
        oidx=new_oidx,
        lidx=new_lidx
    )

    return order_result, new_state
```

## order_nb
创建订单对象。

参数
- `size`：float = np.inf，订单大小：要交易的数量或金额
- `price`：float = np.inf，订单价格：每单位的交易价格
- `size_type`：int = SizeType.Amount，大小类型：订单大小的解释方式
- `direction`：int = Direction.Both，允许方向：订单允许的交易方向
- `fees`：float = 0.0，手续费率：按订单价值的百分比收费
- `fixed_fees`：float = 0.0，固定手续费：每笔订单的固定费用
- `slippage`：float = 0.0，滑点率：价格滑动的百分比
- `min_size`：float = 0.0，最小大小：订单的最小允许大小
- `max_size`：float = np.inf，最大大小：订单的最大允许大小
- `size_granularity`：float = np.nan，大小粒度：订单大小的最小调整单位
- `reject_prob`：float = 0.0，拒绝概率：随机拒绝订单的概率
- `lock_cash`：bool = False，锁定现金：做空时是否锁定现金
- `allow_partial`：bool = True，允许部分成交：是否接受部分填充
- `raise_reject`：bool = False，拒绝异常：拒绝时是否抛出异常
- `log`：bool = False，日志记录：是否记录此订单的详细日志

```python
@njit(cache=True)
def order_nb(size: float = np.nan,
             price: float = np.inf,
             size_type: int = SizeType.Amount,
             direction: int = Direction.Both,
             fees: float = 0.,
             fixed_fees: float = 0.,
             slippage: float = 0.,
             min_size: float = 0.,
             max_size: float = np.inf,
             size_granularity: float = np.nan,
             reject_prob: float = 0.,
             lock_cash: bool = False,
             allow_partial: bool = True,
             raise_reject: bool = False,
             log: bool = False) -> Order:

    return Order(
        size=float(size),
        price=float(price),
        size_type=int(size_type),
        direction=int(direction),
        fees=float(fees),
        fixed_fees=float(fixed_fees),
        slippage=float(slippage),
        min_size=float(min_size),
        max_size=float(max_size),
        size_granularity=float(size_granularity),
        reject_prob=float(reject_prob),
        lock_cash=bool(lock_cash),
        allow_partial=bool(allow_partial),
        raise_reject=bool(raise_reject),
        log=bool(log)
    )

```

## close_position_nb
使用 `order_nb` 创建平仓订单。
- 无论当前是多头还是空头头寸，都会创建完全平掉当前持仓的订单。

参数
- `price`：float，可选 (默认: np.inf)，平仓价格，np.inf 表示市价平仓
- `fees`：float, 可选 (默认: 0.0)，比例手续费率
- `fixed_fees`：float, 可选 (默认: 0.0)，固定手续费
- `slippage`：float, 可选 (默认: 0.0)，滑点率
- `min_size`：float, 可选 (默认: 0.0)，最小订单大小
- `max_size`：float, 可选 (默认: np.inf)，最大订单大小
- `size_granularity`：float, 可选 (默认: np.nan)，订单数量粒度
- `reject_prob`：float, 可选 (默认: 0.0)，随机拒绝概率
- `lock_cash`：bool, 可选 (默认: False)，是否锁定现金
- `allow_partial`：bool, 可选 (默认: True)，是否允许部分成交
- `raise_reject`：bool, 可选 (默认: False)，订单被拒绝时是否抛出异常
- `log`：bool, 可选 (默认: False)，是否记录详细日志
    
返回：Order，目标持仓为 0 的订单对象

```python
@njit(cache=True)
def close_position_nb(price: float = np.inf,
                      fees: float = 0.,
                      fixed_fees: float = 0.,
                      slippage: float = 0.,
                      min_size: float = 0.,
                      max_size: float = np.inf,
                      size_granularity: float = np.nan,
                      reject_prob: float = 0.,
                      lock_cash: bool = False,
                      allow_partial: bool = True,
                      raise_reject: bool = False,
                      log: bool = False) -> Order:

    return order_nb(
        size=0.,
        price=price,
        size_type=SizeType.TargetAmount,
        direction=Direction.Both,
        fees=fees,
        fixed_fees=fixed_fees,
        slippage=slippage,
        min_size=min_size,
        max_size=max_size,
        size_granularity=size_granularity,
        reject_prob=reject_prob,
        lock_cash=lock_cash,
        allow_partial=allow_partial,
        raise_reject=raise_reject,
        log=log
    )
```

## order_nothing_nb
创建空订单：返回一个预定义的空订单对象。

```python
@njit(cache=True)
def order_nothing_nb() -> Order:
    return NoOrder
```

# 参数检查

## check_group_lens_nb
检查资产分组长度数组的有效性：验证 `group_lens` 数组的总和是否等于总列数 `n_cols`，确保资产分组配置正确。

参数
- `group_lens`：Array1d
  - 各组的列数数组，每个元素表示对应组包含的资产数量
  - 例如：[2, 3, 1] 表示第1组有2个资产，第2组有3个资产，第3组有1个资产
- `n_cols`：int
    总列数（总资产数量）

```python
@njit(cache=True)
def check_group_lens_nb(group_lens: tp.Array1d, n_cols: int) -> None:
    if np.sum(group_lens) != n_cols:
        raise ValueError("group_lens has incorrect total number of columns")
```

## check_group_init_cash_nb
检查初始现金配置的有效性

验证初始现金数组的长度是否与现金共享模式和资产分组配置匹配。
这确保了每个资产或资产组都有正确的初始现金分配。

参数
- `group_lens`：Array1d，各组的列数数组
- `n_cols`：int，总列数（总资产数量）
- `init_cash`：Array1d，初始现金数组
- `cash_sharing`：bool，现金共享模式标志
  - `True`：组内资产共享现金，`init_cash` 长度应等于组数
  - `False`：每个资产独立现金，`init_cash` 长度应等于资产数

```python
@njit(cache=True)
def check_group_init_cash_nb(group_lens: tp.Array1d, n_cols: int, init_cash: tp.Array1d, cash_sharing: bool) -> None:
    if cash_sharing:
        if len(init_cash) != len(group_lens):
            raise ValueError("If cash sharing is enabled, init_cash must match the number of groups")
    else:
        if len(init_cash) != n_cols:
            raise ValueError("If cash sharing is disabled, init_cash must match the number of columns")
```

## is_grouped_nb
判断是否存在包含多个资产的组。
    
参数
- `group_lens`：Array1d，各组的列数数组，每个元素表示对应组的资产数量
        

```python
@njit(cache=True)
def is_grouped_nb(group_lens: tp.Array1d) -> bool:
    return np.any(group_lens > 1)
```

# 调用序列管理

## shuffle_call_seq_nb
根据 `group_lens` 随机打乱调用序列数组 `call_seq`。
    
参数
- `call_seq`：Array2d，调用序列数组，形状为(时间步, 资产)
  - 每行表示在该时间步的资产调用顺序
- `group_lens`：Array1d 各组的长度数组
  - 每个元素表示对应组包含的资产数量
    
就地修改：直接修改传入的 `call_seq` 数组，不返回新数组

```python
@njit(cache=True)
def shuffle_call_seq_nb(call_seq: tp.Array2d, group_lens: tp.Array1d) -> None:
    from_col = 0
    # 遍历每个资产组
    for group in range(len(group_lens)):
        to_col = from_col + group_lens[group]
        # 对每个时间步在当前组内进行随机打乱
        for i in range(call_seq.shape[0]):
            np.random.shuffle(call_seq[i, from_col:to_col]) # 组内随机打乱
        # 移动到下一组
        from_col = to_col
```

In [ ]:
import numpy as np
np.random.seed(42)  # 设置随机种子便于重现
from vectorbt.portfolio.nb import shuffle_call_seq_nb
 
# 创建初始调用序列：3个时间步，6个资产，分为2组[2,4]
call_seq = np.array([[0, 1, 2, 3, 4, 5],
                     [0, 1, 2, 3, 4, 5],
                     [0, 1, 2, 3, 4, 5]])
group_lens = np.array([2, 4])  # 第1组2个资产(0,1)，第2组4个资产(2,3,4,5)
 
print("打乱前:")
print(call_seq)
shuffle_call_seq_nb(call_seq, group_lens)
print("打乱后:")
print(call_seq)  # 组内顺序被随机打乱，但组间边界保持

## build_call_seq_nb
构建新的调用序列数组。

参数
- `target_shape`：Shape，目标形状 (时间步数, 资产数)
- `group_lens`：Array1d，各组的长度数组
- `call_seq_type`：int，可选 (默认: CallSeqType.Default)，调用序列类型
  - `CallSeqType.Default`：正常顺序 (0, 1, 2, ...)
  - `CallSeqType.Reversed`：反向顺序 (最后一个资产优先)
  - `CallSeqType.Random`：随机顺序
    
返回：Array2d，调用序列数组，形状为 `target_shape`

```python
@njit(cache=True)
def build_call_seq_nb(target_shape: tp.Shape,
                      group_lens: tp.Array1d,
                      call_seq_type: int = CallSeqType.Default) -> tp.Array2d:
    if call_seq_type == CallSeqType.Reversed:
        out = np.full(target_shape[1], 1, dtype=np.int64)
        out[np.cumsum(group_lens)[1:] - group_lens[1:] - 1] -= group_lens[1:]
        out = np.cumsum(out[::-1])[::-1] - 1
        out = out * np.ones((target_shape[0], 1), dtype=np.int64)
        return out
    out = np.full(target_shape[1], 1, dtype=np.int64)
    out[np.cumsum(group_lens)[:-1]] -= group_lens[:-1]
    out = np.cumsum(out) - 1
    out = out * np.ones((target_shape[0], 1), dtype=np.int64)
    if call_seq_type == CallSeqType.Random:
        shuffle_call_seq_nb(out, group_lens)
    return out
```

In [ ]:
from vectorbt.portfolio.nb import build_call_seq_nb, CallSeqType

# 创建3个时间步，6个资产，分2组[3,3]的调用序列
target_shape = (3, 6)
group_lens = np.array([3, 3])
 
# 默认顺序
default_seq = build_call_seq_nb(target_shape, group_lens, CallSeqType.Default)
print("默认顺序:")
print(default_seq)
 
# 反向顺序
reversed_seq = build_call_seq_nb(target_shape, group_lens, CallSeqType.Reversed)
print("反向顺序:")
print(reversed_seq)

# 随机顺序
np.random.seed(42)
random_seq = build_call_seq_nb(target_shape, group_lens, CallSeqType.Random)
print("随机顺序:")
print(random_seq)  # 组内随机排列

## require_call_seq
确保调用序列数组 `call_seq` 具有正确的数据类型和内存布局，来满足后续 Numba 编译函数的严格要求

参数
- `call_seq`：Array2d，调用序列数组
    
返回：`Array2d`，满足要求的调用序列数组
    
内存要求
- `dtype=np.int64`：64 位整数类型
- requirements
  - `'A'`：对齐 (Aligned)
  - `'O'`：拥有数据 (Owndata)  
  - `'W'`：可写 (Writeable)
  - `'F'`：Fortran 风格连续

```python
def require_call_seq(call_seq: tp.Array2d) -> tp.Array2d:
    return np.require(call_seq, dtype=np.int64, requirements=['A', 'O', 'W', 'F'])
```

## build_call_seq
构建调用序列数组（非编译优化版本），`build_call_seq_nb` 的非编译版本。
- 使用 NumPy 的向量化操作实现更快的执行速度，以满足后续 Numba 编译函数的要求。

参数
- `target_shape`：Shape，目标形状 (时间步数, 资产数)
- `group_lens`：Array1d，各组的长度数组
- `call_seq_type`：int, 可选 (默认: CallSeqType.Default)，调用序列类型
  - `CallSeqType.Default`：正常顺序 (0, 1, 2, ...)
  - `CallSeqType.Reversed`：反向顺序 (最后一个资产优先)
  - `CallSeqType.Random`：随机顺序
    
返回：Array2d，调用序列数组，满足内存和类型要求

```python
def build_call_seq(target_shape: tp.Shape,
                   group_lens: tp.Array1d,
                   call_seq_type: int = CallSeqType.Default) -> tp.Array2d:
    call_seq = np.full(target_shape[1], 1, dtype=np.int64)
    if call_seq_type == CallSeqType.Reversed:
        call_seq[np.cumsum(group_lens)[1:] - group_lens[1:] - 1] -= group_lens[1:]
        call_seq = np.cumsum(call_seq[::-1])[::-1] - 1
    else:
        call_seq[np.cumsum(group_lens[:-1])] -= group_lens[:-1]
        call_seq = np.cumsum(call_seq) - 1
    call_seq = np.broadcast_to(call_seq, target_shape)
    if call_seq_type == CallSeqType.Random:
        call_seq = require_call_seq(call_seq)
        shuffle_call_seq_nb(call_seq, group_lens)
    return require_call_seq(call_seq)
```

# 辅助工具函数

## get_col_elem_nb
根据上下文对象 `ctx` 和列索引 `col` 从数组 `a` 获取对应元素。

参数
- `ctx`：`Union[OrderContext, PostOrderContext, SignalContext]`，上下文对象，包含：
  - `i`：当前时间步索引
  - `flex_2d`：是否启用2D灵活索引
- `col`：int，目标列索引（资产索引）
- `a`：ArrayLike，待查询的数组
    
返回：Scalar，指定位置的元素值

```python
@njit(cache=True)
def get_col_elem_nb(ctx: tp.Union[RowContext, SegmentContext, FlexOrderContext], col: int,
                    a: tp.ArrayLike) -> tp.Scalar:
    return flex_select_auto_nb(a, ctx.i, col, ctx.flex_2d)
```

## get_elem_nb
根据上下文对象 `ctx` 从数组 `a` 获取对应元素。

参数
- `ctx`：`Union[OrderContext, PostOrderContext, SignalContext]`，上下文对象，包含：
  - `i`：当前时间步索引
  - `col`：当前列索引（资产索引）
  - `flex_2d`：是否启用2D灵活索引
- `a`：ArrayLike，待查询的数组
    
返回：Scalar，指定位置的元素值

```python
@njit(cache=True)
def get_elem_nb(ctx: tp.Union[OrderContext, PostOrderContext, SignalContext],
                a: tp.ArrayLike) -> tp.Scalar:
    return flex_select_auto_nb(a, ctx.i, ctx.col, ctx.flex_2d)
```

## get_group_value_nb
计算指定资产组的当前总价值，包括现金和所有持仓资产的市值。
- $cash\_now{\rm{ }} + \sum\limits_{i \in \left[ {from\_col,to\_col} \right)} {last\_position\left[ i \right]last\_val\_price\left[ i \right]}$

参数
- `from_col`：int，资产组起始列索引（包含）
- `to_col`：int，资产组结束列索引（不包含）
- `cash_now`：float，当前现金余额
- `last_position`：Array1d，最新的持仓数量数组，每个元素对应一个资产
- `last_val_price`：Array1d，最新的估值价格数组，每个元素对应一个资产的当前价格
    
返回：float，资产组的总价值 = 现金 + 所有持仓的市值总和

```python
@njit(cache=True)
def get_group_value_nb(from_col: int,
                       to_col: int,
                       cash_now: float,
                       last_position: tp.Array1d,
                       last_val_price: tp.Array1d) -> float:
    group_value = cash_now
    group_len = to_col - from_col
    for k in range(group_len):
        col = from_col + k
        if last_position[col] != 0:
            group_value += last_position[col] * last_val_price[col]
    return group_value
```

## get_group_value_ctx_nb
从上下文获取资产组价值，`get_group_value_nb` 的上下文版本。
- 自动从 `SegmentContext` 中提取所需的参数来计算资产组价值。

参数
- `seg_ctx`：SegmentContext，分段上下文对象，包含
  - `from_col, to_col`：当前资产组的列范围
  - `group`：当前资产组索引
  - `last_cash`：各组的最新现金数组
  - `last_position`：最新持仓数组
  - `last_val_price`：最新估值价格数组
  - `cash_sharing`：现金共享标志
    
返回：float，当前资产组的总价值

```python
@njit(cache=True)
def get_group_value_ctx_nb(seg_ctx: SegmentContext) -> float:
    if not seg_ctx.cash_sharing:
        raise ValueError("Cash sharing must be enabled")
    return get_group_value_nb(
        seg_ctx.from_col,
        seg_ctx.to_col,
        seg_ctx.last_cash[seg_ctx.group],
        seg_ctx.last_position,
        seg_ctx.last_val_price
    )
```

## approx_order_value_nb
根据订单参数和当前状态，估算订单的价值（所需现金量）。

参数
- `size`：float，订单大小，含义取决于 `size_type`
- `size_type`：int，订单大小类型，参见 `SizeType` 枚举：
  - `Amount`：具体数量。返回 `size * val_price_now`
  - `Value`：价值金额。返回 `size`
  - `Percent`：百分比（现金或持仓的百分比）
    - 如果 `size >= 0`（此时 `size` 表示占据当前现金余额的百分比）
      - 返回 `size * cash_now`
    - 否则
      - 如果 `direction == Direction.LongOnly`（此时 `size` 表示占据当前该资产价值的百分比）
        - 返回 `size * position_now * val_price_now`
      - 否则
        - 返回 `size * (2 * max(position_now * val_price_now, 0) + max(free_cash_now, 0))`
  - `TargetAmount`：目标持仓数量。
    - 此时 `size` 表示：加上该订单后，持有的该资产总数
    - 返回 `size * val_price_now - position_now * val_price_now`
  - `TargetValue`：目标持仓价值。
    - 此时 `size` 表示：加上该订单后，持有的该资产总价值
    - 返回 `size - position_now * val_price_now`
  - `TargetPercent`：目标持仓占组合的百分比。
    - 返回 `size * value_now - position_now * val_price_now`
- `direction`：int，交易方向限制，参见 `Direction` 枚举：
  - `LongOnly`：int = 0，仅多头：只允许多头（买入持有）仓位
  - `ShortOnly`：int = 1，仅空头：只允许空头（卖空）仓位  
  - `Both`：int = 2，双向：允许多头和空头仓位
- `cash_now`：float，当前现金余额
- `position_now`：float，当前持仓数量
- `free_cash_now`：float，当前可用现金
- `val_price_now`：float，当前估值价格
- `value_now`：float，当前投资组合总价值
  
返回：float， 估算的订单价值（所需现金）
- 正值表示买入，负值表示卖出，如果无法计算则返回 NaN

```python
@njit(cache=True)
def approx_order_value_nb(size: float,
                          size_type: int,
                          direction: int,
                          cash_now: float,
                          position_now: float,
                          free_cash_now: float,
                          val_price_now: float,
                          value_now: float) -> float:
    if direction == Direction.ShortOnly:
        size *= -1
    asset_value_now = position_now * val_price_now
    if size_type == SizeType.Amount:
        return size * val_price_now
    if size_type == SizeType.Value:
        return size
    if size_type == SizeType.Percent:
        if size >= 0:
            return size * cash_now
        else:
            if direction == Direction.LongOnly:
                return size * asset_value_now
            return size * (2 * max(asset_value_now, 0) + max(free_cash_now, 0))
    if size_type == SizeType.TargetAmount:
        return size * val_price_now - asset_value_now
    if size_type == SizeType.TargetValue:
        return size - asset_value_now
    if size_type == SizeType.TargetPercent:
        return size * value_now - asset_value_now
    return np.nan
```

## sort_call_seq_out_nb
基于订单价值对调用序列 `call_seq_out` 进行就地排序。

参数
- `ctx`：`SegmentContext`，分段上下文对象，包含当前状态信息
  - `target_shape`：tp.Shape，模拟目标形状
  - `group_lens`：tp.Array1d，每组列数
  - `init_cash`：tp.Array1d，初始资金
  - `cash_sharing`：bool，现金共享标志
  - `call_seq`：tp.Optional[tp.Array2d]，调用序列
  - `segment_mask`：tp.ArrayLike，段掩码
  - `call_pre_segment`：bool，调用段前函数标志
  - `call_post_segment`：bool，调用段后函数标志
  - `close`：tp.ArrayLike，收盘价数据
  - `ffill_val_price`：bool，前向填充估值价格标志
  - `update_value`：bool，更新价值标志
  - `fill_pos_record`：bool，填充仓位记录标志
  - `flex_2d`：bool，灵活二维索引标志
  - `order_records`：tp.RecordArray，订单记录数组
  - `log_records`：tp.RecordArray，日志记录数组
  - `last_cash`：tp.Array1d，最新现金状态
  - `last_position`：tp.Array1d，最新仓位状态
  - `last_debt`：tp.Array1d，最新债务状态
  - `last_free_cash`：tp.Array1d，最新可用现金
  - `last_val_price`：tp.Array1d，最新估值价格
  - `last_value`：tp.Array1d，最新组合价值
  - `second_last_value`：tp.Array1d，次新组合价值
  - `last_return`：tp.Array1d，最新收益率
  - `last_oidx`：tp.Array1d，最新订单索引
  - `last_lidx`：tp.Array1d，最新日志索引
  - `last_pos_record`：tp.RecordArray，最新仓位记录
  - `group`：int，当前组索引
  - `group_len`：int，当前组大小
  - `from_col`：int，组起始列索引
  - `to_col`：int，组结束列索引+1
  - `i`：int，当前行索引
  - `call_seq_now`：tp.Optional[tp.Array1d]，当前段内的调用序列
- `size`：ArrayLike，订单大小数组，支持灵活索引 
- `size_type`：ArrayLike，订单大小类型数组，支持灵活索引
- `direction`：ArrayLike，交易方向数组，支持灵活索引
- `order_value_out`：Array1d，输出的订单价值数组，长度应匹配组内资产数量
  - 函数执行前应为空，执行后包含排序后的订单价值
- `call_seq_out`：Array1d，输入/输出的调用序列数组，长度应匹配组内资产数量
  - 输入时应按默认顺序填充 (0, 1, 2, ...)；输出时为按价值排序后的资产索引序列
- `ctx_select`：bool, 可选 (默认: True)，索引选择模式
    - True：使用 `get_col_elem_nb` 进行上下文选择
    - False：使用 `flex_select_auto_nb` 进行灵活选择

```python
@njit(cache=True)
def sort_call_seq_out_nb(ctx: SegmentContext,
                         size: tp.ArrayLike,
                         size_type: tp.ArrayLike,
                         direction: tp.ArrayLike,
                         order_value_out: tp.Array1d,
                         call_seq_out: tp.Array1d,
                         ctx_select: bool = True) -> None:
    if not ctx.cash_sharing:
        raise ValueError("Cash sharing must be enabled")
    size_arr = np.asarray(size)
    size_type_arr = np.asarray(size_type)
    direction_arr = np.asarray(direction)

    # 获取当前组合总价值
    group_value_now = get_group_value_ctx_nb(ctx)
    group_len = ctx.to_col - ctx.from_col # 组内资产数量
    # 遍历组内每个资产，计算订单价值
    for k in range(group_len):
        if call_seq_out[k] != k:
            raise ValueError("call_seq_out should follow CallSeqType.Default")
        # 当前资产的绝对列索引
        col = ctx.from_col + k
        if ctx_select:
            _size = get_col_elem_nb(ctx, col, size_arr)
            _size_type = get_col_elem_nb(ctx, col, size_type_arr)
            _direction = get_col_elem_nb(ctx, col, direction_arr)
        else:
            _size = flex_select_auto_nb(size_arr, k, 0, False)
            _size_type = flex_select_auto_nb(size_type_arr, k, 0, False)
            _direction = flex_select_auto_nb(direction_arr, k, 0, False)
        # 获取现金状态（现金共享模式使用组级现金）
        if ctx.cash_sharing:
            cash_now = ctx.last_cash[ctx.group]
            free_cash_now = ctx.last_free_cash[ctx.group]
        else:
            cash_now = ctx.last_cash[col]
            free_cash_now = ctx.last_free_cash[col]
        # 计算该资产的近似订单价值
        order_value_out[k] = approx_order_value_nb(
            _size,
            _size_type,
            _direction,
            cash_now,
            ctx.last_position[col],
            free_cash_now,
            ctx.last_val_price[col],
            group_value_now
        )
    # 根据订单价值对调用序列进行排序
    # 使用插入排序算法，按价值从大到小排序
    insert_argsort_nb(order_value_out, call_seq_out)
```

## sort_call_seq_nb

```python
@njit(cache=True)
def sort_call_seq_nb(ctx: SegmentContext,
                     size: tp.ArrayLike,
                     size_type: tp.ArrayLike,
                     direction: tp.ArrayLike,
                     order_value_out: tp.Array1d,
                     ctx_select: bool = True) -> None:
    if ctx.call_seq_now is None:
        raise ValueError("Call sequence array is None. Use sort_call_seq_out_nb to sort a custom array.")
    sort_call_seq_out_nb(
        ctx,
        size,
        size_type,
        direction,
        order_value_out,
        ctx.call_seq_now,
        ctx_select=ctx_select
    )
```

## replace_inf_price_nb

```python
@njit(cache=True)
def replace_inf_price_nb(prev_close: float, close: float, order: Order) -> Order:
    order_price = order.price
    if order_price > 0:
        order_price = close  # upper bound is close
    else:
        order_price = prev_close  # lower bound is prev close
    return order_nb(
        size=order.size,
        price=order_price,
        size_type=order.size_type,
        direction=order.direction,
        fees=order.fees,
        fixed_fees=order.fixed_fees,
        slippage=order.slippage,
        min_size=order.min_size,
        max_size=order.max_size,
        size_granularity=order.size_granularity,
        reject_prob=order.reject_prob,
        lock_cash=order.lock_cash,
        allow_partial=order.allow_partial,
        raise_reject=order.raise_reject,
        log=order.log
    )
```

## try_order_nb

```python
@njit(cache=True)
def try_order_nb(ctx: OrderContext, order: Order) -> tp.Tuple[ExecuteOrderState, OrderResult]:
    state = ProcessOrderState(
        cash=ctx.cash_now,
        position=ctx.position_now,
        debt=ctx.debt_now,
        free_cash=ctx.free_cash_now,
        val_price=ctx.val_price_now,
        value=ctx.value_now,
        oidx=-1,
        lidx=-1
    )
    if np.isinf(order.price):
        if ctx.i > 0:
            prev_close = flex_select_auto_nb(ctx.close, ctx.i - 1, ctx.col, ctx.flex_2d)
        else:
            prev_close = np.nan
        close = flex_select_auto_nb(ctx.close, ctx.i, ctx.col, ctx.flex_2d)
        order = replace_inf_price_nb(prev_close, close, order)
    return execute_order_nb(state, order)
```

## init_records_nb

```python
@njit(cache=True)
def init_records_nb(target_shape: tp.Shape,
                    max_orders: tp.Optional[int] = None,
                    max_logs: int = 0) -> tp.Tuple[tp.RecordArray, tp.RecordArray]:
    if max_orders is None:
        _max_orders = target_shape[0] * target_shape[1]
    else:
        _max_orders = max_orders
    order_records = np.empty(_max_orders, dtype=order_dt)
    if max_logs == 0:
        max_logs = 1
    log_records = np.empty(max_logs, dtype=log_dt)
    return order_records, log_records
```

## update_open_pos_stats_nb

```python
@njit(cache=True)
def update_open_pos_stats_nb(record: tp.Record, position_now: float, price: float) -> None:
    if record['id'] >= 0 and record['status'] == TradeStatus.Open:
        if np.isnan(record['exit_price']):
            exit_price = price
        else:
            exit_size_sum = record['size'] - abs(position_now)
            exit_gross_sum = exit_size_sum * record['exit_price']
            exit_gross_sum += abs(position_now) * price
            exit_price = exit_gross_sum / record['size']
        pnl, ret = get_trade_stats_nb(
            record['size'],
            record['entry_price'],
            record['entry_fees'],
            exit_price,
            record['exit_fees'],
            record['direction']
        )
        record['pnl'] = pnl
        record['return'] = ret
```

## update_pos_record_nb

```python
@njit(cache=True)
def update_pos_record_nb(record: tp.Record,
                         i: int,
                         col: int,
                         position_before: float,
                         position_now: float,
                         order_result: OrderResult) -> None:
    if order_result.status == OrderStatus.Filled:
        if position_before == 0 and position_now != 0:
            # New position opened
            record['id'] += 1
            record['col'] = col
            record['size'] = order_result.size
            record['entry_idx'] = i
            record['entry_price'] = order_result.price
            record['entry_fees'] = order_result.fees
            record['exit_idx'] = -1
            record['exit_price'] = np.nan
            record['exit_fees'] = 0.
            if order_result.side == OrderSide.Buy:
                record['direction'] = TradeDirection.Long
            else:
                record['direction'] = TradeDirection.Short
            record['status'] = TradeStatus.Open
            record['parent_id'] = record['id']
        elif position_before != 0 and position_now == 0:
            # Position closed
            record['exit_idx'] = i
            if np.isnan(record['exit_price']):
                exit_price = order_result.price
            else:
                exit_size_sum = record['size'] - abs(position_before)
                exit_gross_sum = exit_size_sum * record['exit_price']
                exit_gross_sum += abs(position_before) * order_result.price
                exit_price = exit_gross_sum / record['size']
            record['exit_price'] = exit_price
            record['exit_fees'] += order_result.fees
            pnl, ret = get_trade_stats_nb(
                record['size'],
                record['entry_price'],
                record['entry_fees'],
                record['exit_price'],
                record['exit_fees'],
                record['direction']
            )
            record['pnl'] = pnl
            record['return'] = ret
            record['status'] = TradeStatus.Closed
        elif np.sign(position_before) != np.sign(position_now):
            # Position reversed
            record['id'] += 1
            record['size'] = abs(position_now)
            record['entry_idx'] = i
            record['entry_price'] = order_result.price
            new_pos_fraction = abs(position_now) / abs(position_now - position_before)
            record['entry_fees'] = new_pos_fraction * order_result.fees
            record['exit_idx'] = -1
            record['exit_price'] = np.nan
            record['exit_fees'] = 0.
            if order_result.side == OrderSide.Buy:
                record['direction'] = TradeDirection.Long
            else:
                record['direction'] = TradeDirection.Short
            record['status'] = TradeStatus.Open
            record['parent_id'] = record['id']
        else:
            # Position changed
            if abs(position_before) <= abs(position_now):
                # Position increased
                entry_gross_sum = record['size'] * record['entry_price']
                entry_gross_sum += order_result.size * order_result.price
                entry_price = entry_gross_sum / (record['size'] + order_result.size)
                record['entry_price'] = entry_price
                record['entry_fees'] += order_result.fees
                record['size'] += order_result.size
            else:
                # Position decreased
                if np.isnan(record['exit_price']):
                    exit_price = order_result.price
                else:
                    exit_size_sum = record['size'] - abs(position_before)
                    exit_gross_sum = exit_size_sum * record['exit_price']
                    exit_gross_sum += order_result.size * order_result.price
                    exit_price = exit_gross_sum / (exit_size_sum + order_result.size)
                record['exit_price'] = exit_price
                record['exit_fees'] += order_result.fees

        # Update open position stats
        update_open_pos_stats_nb(
            record,
            position_now,
            order_result.price
        )
```

# 投资组合模拟

## simulate_from_orders_nb

## generate_stop_signal_nb

## resolve_stop_price_and_slippage_nb

## resolve_signal_conflict_nb

## resolve_dir_conflict_nb

## resolve_opposite_entry_nb

## signals_to_size_nb

## should_update_stop_nb

## get_stop_price_nb

## no_signal_func_nb

## no_adjust_sl_func_nb

## no_adjust_tp_func_nb

## simulate_from_signal_func_nb

## dir_enex_signal_func_nb

## ls_enex_signal_func_nb

## no_pre_func_nb

## no_order_func_nb

## no_post_func_nb

## simulate_nb

## simulate_row_wise_nb

## no_flex_order_func_nb

## flex_simulate_nb

## flex_simulate_row_wise_nb

# 交易记录处理

## get_trade_stats_nb

## fill_trade_record_nb

## fill_entry_trades_in_position_nb

## get_entry_trades_nb

## get_exit_trades_nb

## trade_winning_streak_nb

## trade_losing_streak_nb

# 仓位记录

## fill_position_record_nb

## copy_trade_record_nb

## get_positions_nb

# 资产持仓

## get_long_size_nb

## get_short_size_nb

## asset_flow_nb

## assets_nb

## i_group_any_reduce_nb

## position_mask_grouped_nb

## group_mean_reduce_nb

## position_coverage_grouped_nb

# 资金管理

## get_free_cash_diff_nb

## cash_flow_nb

## sum_grouped_nb

## cash_flow_grouped_nb

## init_cash_grouped_nb

## init_cash_nb

## cash_nb

## cash_in_sim_order_nb

## cash_grouped_nb

# 绩效分析

## asset_value_nb

## asset_value_grouped_nb

## value_in_sim_order_nb

## value_nb

## total_profit_nb

## total_profit_grouped_nb

## final_value_nb

## total_return_nb

## returns_in_sim_order_nb

## asset_returns_nb

## benchmark_value_nb

## benchmark_value_grouped_nb

## total_benchmark_return_nb

## gross_exposure_nb